## Starting phase

importing everything & performing data cleaning and some other stuff... 

In [34]:
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import  DataLoader, TensorDataset


%matplotlib inline

In [35]:
data = pd.read_csv("ct_slice_train.csv")
data.head()

,ID,value0,value1,value2,value3,value4,value5,value6,value7,value8,...,value375,value376,value377,value378,value379,value380,value381,value382,value383,TARGET
0,0,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.673369,0.000000,-0.25,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.25,56.667181
1,1,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.25,62.455880
2,2,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.561189,0.791667,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.25,57.597645
3,3,0.0,0.0,0.019536,0.0,0.640917,0.875338,0.000000,0.000000,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,75.635799
4,4,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.25,54.977994


In [36]:
data.drop(columns=['ID'],inplace=True)

In [37]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42800 entries, 0 to 42799
Columns: 385 entries, value0 to TARGET
dtypes: float64(385)
memory usage: 125.7 MB


In [38]:
data.isnull().any(axis=0).sum()

0

In [39]:
data["TARGET"].describe()

count    42800.000000
mean        47.086058
std         22.325819
min          1.738733
25%         30.005851
50%         44.060878
75%         63.752913
max         97.320148
Name: TARGET, dtype: float64

In [40]:
data.shape

(42800, 385)

In [41]:
X = data.drop(columns=["TARGET"])
y = data['TARGET']

X.shape, y.shape

((42800, 384), (42800,))

In [42]:
zero_columns = X.columns[X.var(axis=0) < 0.001]  # ['value59', 'value69', 'value179', 'value189', 'value351']
X = X.drop(columns=zero_columns)
X.shape

(42800, 374)

## Device Agnostic Code

In [43]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

## Train & Test split

In [44]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.20, random_state=42)

In [45]:
len(X_train), len(X_val), len(y_train), len(y_val)

(34240, 8560, 34240, 8560)

In [46]:
from sklearn.preprocessing import StandardScaler

scalar = StandardScaler()
X_train = scalar.fit_transform(X_train)
X_val = scalar.transform(X_val)

In [47]:
scalar_1 = StandardScaler()
y_train = scalar_1.fit_transform(y_train.values.reshape(-1, 1))
y_val = scalar_1.transform(y_val.values.reshape(-1, 1))

In [48]:
print(type(X_train), type(y_train))
print(type(X_val), type(y_val))

<class 'numpy.ndarray'> <class 'numpy.ndarray'>
<class 'numpy.ndarray'> <class 'numpy.ndarray'>


In [49]:
X_train = torch.tensor(X_train, dtype=torch.float32, device=device)
X_val = torch.tensor(X_val, dtype=torch.float32, device=device)
y_train = torch.tensor(y_train, dtype=torch.float32, device=device)
y_val = torch.tensor(y_val, dtype=torch.float32, device=device)

In [50]:
train_data = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
val_data = DataLoader(TensorDataset(X_val, y_val), batch_size=64, shuffle=False)
print(f"Total number of batches: 34,240 samples ÷ 64 samples per batch = {len(train_data)}") 

Total number of batches: 34,240 samples ÷ 64 samples per batch = 535


## Visualization

In [51]:
def plot_prediction(train_data=X_train,
                    train_labels=y_train,
                    test_data=X_val,
                    test_labels=y_val,
                    predictions=None):
    
    plt.figure(figsize=(10, 7))
    plt.scatter(train_data, train_labels, c='b', s=4, label="Training data")
    plt.scatter(test_data, test_labels, c='g', s=4, label="Testing data")

    if predictions is not None:
        plt.scatter(test_data, predictions, c='r', s=4, label="predictions")

    plt.legend(prop={"size": 14});
    

In [52]:
len(X_val), len(y_val)

(8560, 8560)

In [53]:
# plot_prediction()

## Model Building

In [54]:
torch.manual_seed(42)

class LinearRegressionModel(nn.Module):
    
    def __init__(self, in_features, hidden_sizes, out_features):
        super().__init__()
        sizes = [in_features] + hidden_sizes
        layers = []
        for i in range(len(sizes)-1):
            layers.append(nn.Linear(sizes[i], sizes[i+1]))
            layers.append(nn.ReLU())
        layers.append(nn.Linear(sizes[-1], out_features))
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(x)
   


In [55]:
torch.manual_seed(42)

model = LinearRegressionModel(374, [256, 128, 64], 1)
model, model.state_dict()

(LinearRegressionModel(
   (net): Sequential(
     (0): Linear(in_features=374, out_features=256, bias=True)
     (1): ReLU()
     (2): Linear(in_features=256, out_features=128, bias=True)
     (3): ReLU()
     (4): Linear(in_features=128, out_features=64, bias=True)
     (5): ReLU()
     (6): Linear(in_features=64, out_features=1, bias=True)
   )
 ),
 OrderedDict([('net.0.weight',
               tensor([[ 3.9533e-02,  4.2919e-02, -1.2114e-02,  ...,  2.9832e-02,
                        -3.0107e-02, -6.7122e-03],
                       [-3.8119e-02, -2.4946e-02,  9.3644e-03,  ..., -2.3476e-02,
                         3.6169e-02, -3.5381e-02],
                       [-2.8513e-02,  3.7747e-02,  1.6315e-02,  ...,  1.8254e-02,
                        -2.6432e-02, -4.2253e-02],
                       ...,
                       [ 3.9516e-02,  4.5252e-02,  3.3498e-02,  ...,  3.9001e-02,
                         1.5738e-02,  7.3761e-05],
                       [ 3.0505e-02,  2.4870e-02, -5.27

In [56]:
next(model.parameters()).device

device(type='cpu')

In [57]:
model.to(device)
next(model.parameters()).device

device(type='cpu')

## Training

In [58]:
Loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(params=model.parameters(),
                            lr=0.001)

In [ ]:
torch.manual_seed(42)

epochs = 1000
TRAINING_LOSS = []
VALIDATION_LOSS = []

for epoch in range(epochs):

    model.train()
    for x_batch, y_batch in train_data:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)
        
        y_pred = model(x_batch)
        loss = Loss_fn(y_pred, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    TRAINING_LOSS.append(loss.item())

    # Testing
    model.eval()
    with torch.inference_mode():
        for x_batch, y_batch in val_data:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            
            test_pred = model(x_batch)
            test_loss = Loss_fn(test_pred, y_batch)
            VALIDATION_LOSS.append(test_loss.item())

    # print out what's happening
    if epoch % 10 == 0:
        print(f"Epoch: {epoch} | Loss: {loss} | Test loss: {test_loss}")


Epoch: 0 | Loss: 0.7609257102012634 | Test loss: 0.7439139485359192
Epoch: 10 | Loss: 0.051403775811195374 | Test loss: 0.04697291553020477
Epoch: 20 | Loss: 0.029665054753422737 | Test loss: 0.02605157159268856
Epoch: 30 | Loss: 0.02883528545498848 | Test loss: 0.018816063180565834
Epoch: 40 | Loss: 0.01137616392225027 | Test loss: 0.01540455687791109
Epoch: 50 | Loss: 0.015323775820434093 | Test loss: 0.013431932777166367
Epoch: 60 | Loss: 0.018219927325844765 | Test loss: 0.01223013922572136
Epoch: 70 | Loss: 0.011575683951377869 | Test loss: 0.011605157516896725
Epoch: 80 | Loss: 0.008426729589700699 | Test loss: 0.010861378163099289
Epoch: 90 | Loss: 0.019311828538775444 | Test loss: 0.010334734804928303
Epoch: 100 | Loss: 0.011035997420549393 | Test loss: 0.009837389923632145
Epoch: 110 | Loss: 0.010545746423304081 | Test loss: 0.009282625280320644
Epoch: 120 | Loss: 0.009082687087357044 | Test loss: 0.009067974053323269
Epoch: 130 | Loss: 0.006463498342782259 | Test loss: 0.0086

KeyboardInterrupt: 

## Testing upon external data

In [103]:
ext_test_data = pd.read_csv("ct_slice_test_noLabels.csv")
print(ext_test_data.shape)

(10700, 385)


In [104]:
ext_test_data.head(2)

,ID,value0,value1,value2,value3,value4,value5,value6,value7,value8,...,value374,value375,value376,value377,value378,value379,value380,value381,value382,value383
0,42800,0.0,0.0,0.0,0.0,0.0,0.000000,0.909039,0.0,-0.25,...,0.0,0.0,0.0,0.00000,0.0000,0.000000,0.865903,0.343742,0.0,-0.25
1,42801,0.0,0.0,0.0,0.0,0.0,0.793039,0.000000,0.0,0.00,...,0.0,0.0,0.0,0.99694,0.9998,0.999976,0.998890,0.000000,0.0,-0.25


In [105]:
id_col = ext_test_data["ID"] # Taking it for submission


In [106]:
X1 = ext_test_data.drop(["ID"], axis=1)
X1.head(2)

,value0,value1,value2,value3,value4,value5,value6,value7,value8,value9,...,value374,value375,value376,value377,value378,value379,value380,value381,value382,value383
0,0.0,0.0,0.0,0.0,0.0,0.000000,0.909039,0.0,-0.25,-0.25,...,0.0,0.0,0.0,0.00000,0.0000,0.000000,0.865903,0.343742,0.0,-0.25
1,0.0,0.0,0.0,0.0,0.0,0.793039,0.000000,0.0,0.00,-0.25,...,0.0,0.0,0.0,0.99694,0.9998,0.999976,0.998890,0.000000,0.0,-0.25


In [107]:
col = X1.columns[X1.var(axis=0) < 0.001]
print(len(col))
X1 = X1.drop(columns=col)
X1.shape

10


(10700, 374)

In [108]:
X1 = scalar.transform(X1) 

In [109]:
X1_tensor = torch.tensor(X1, dtype=torch.float32, device=device)

In [110]:
model.eval()
with torch.inference_mode():
    predictions = model(X1_tensor)

In [111]:
predictions.shape

torch.Size([10700, 1])

In [112]:
predictions

tensor([[ 0.1561],
        [-0.3481],
        [ 1.1302],
        ...,
        [-0.8324],
        [-1.2770],
        [ 0.0337]])

In [114]:
predictions.cpu().numpy()
scalar.inverse_transform(predictions)
predictions

ValueError: non-broadcastable output operand with shape (10700,1) doesn't match the broadcast shape (10700,374)